# Ottimizzazione delle prestazioni di una rete neurale per il settore food

Progetto per il corso "Deep Learning applicato con PyTorch" - ProfessionAI

**Obiettivo:** Implementare un sistema di classificazione di immagini nel settore food utilizzando tecniche di Data Augmentation, Transfer Learning, Fine Tuning e Regularizzazione.


## 1. Environment Setup

**Cosa facciamo in questa sezione:**
- Import delle librerie necessarie (PyTorch, torchvision, PIL, matplotlib, sklearn, ecc.)
- Configurazione del device (CPU/GPU) per il training
- Download e estrazione del dataset dal repository remoto

**Termini tecnici:**
- **Device**: Dispositivo di calcolo (CPU o CUDA/GPU) su cui eseguire il training
- **Environment**: Ambiente di sviluppo con tutte le dipendenze installate


In [ ]:
# Importo tutte le librerie che mi serviranno per questo progetto
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import ResNet18_Weights
from PIL import Image
import os
import zipfile
import shutil
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
from collections import Counter
import random
import platform

# Imposto seed per riproducibilità
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

# Controllo se ho una GPU disponibile, altrimenti uso la CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo utilizzato: {device}')


In [ ]:
# Definisco l'URL del dataset e i nomi dei file che userò
dataset_url = 'https://proai-datasets.s3.eu-west-3.amazonaws.com/dataset_food_classification.zip'
dataset_zip = 'dataset_food_classification.zip'
dataset_dir = 'dataset_food_classification'

# Scarico il dataset solo se non l'ho già scaricato prima
if not os.path.exists(dataset_zip):
    print('Download del dataset in corso...')
    import urllib.request
    urllib.request.urlretrieve(dataset_url, dataset_zip)
    print('Download completato!')
else:
    print('Dataset già presente')

# Estraggo il dataset solo se non l'ho già estratto
if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)  # Creo la cartella se non esiste

    print('Estrazione del dataset in corso...')
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)  # Estraggo tutto nella cartella
    print('Estrazione completata!')
else:
    print('Dataset già estratto')


## 2. Data Exploration

**Cosa facciamo in questa sezione:**
- Esplorazione della struttura del dataset (organizzazione in cartelle per classe)
- Conteggio delle immagini per ogni classe per verificare il bilanciamento
- Visualizzazione di campioni rappresentativi per ogni classe

**Termini tecnici:**
- **Data Exploration**: Analisi preliminare del dataset per comprenderne struttura e caratteristiche
- **Class Distribution**: Distribuzione delle immagini tra le diverse classi
- **Class Imbalance**: Squilibrio nel numero di campioni tra classi diverse


In [ ]:
# Esploro il dataset per capire come è organizzato
# Il dataset dovrebbe avere una cartella per ogni classe di cibo

def find_classes(root_dir):
    """Trovo tutte le cartelle che rappresentano le classi"""
    classes = []
    for item in os.listdir(root_dir):
        # Escludo cartelle di sistema come __MACOSX
        if item.startswith('__') or item.startswith('.'):
            continue
        item_path = os.path.join(root_dir, item)
        if os.path.isdir(item_path):
            # Verifico che la cartella contenga immagini
            images = [f for f in os.listdir(item_path)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if len(images) > 0:  # Solo se contiene immagini
                classes.append(item)
    return sorted(classes)

# Trovo la directory principale
possible_paths = [
    os.path.join(dataset_dir, 'dataset', 'train'),  # Provo prima il percorso più comune
    os.path.join(dataset_dir, 'train'),
    os.path.join(dataset_dir, 'data'),
    os.path.join(dataset_dir, 'images'),
    dataset_dir
]

data_root = None
for path in possible_paths:
    if os.path.exists(path):
        classes = find_classes(path)
        if len(classes) > 0:
            data_root = path
            break

# Verifico che il dataset esista
if not os.path.exists(data_root):
    raise FileNotFoundError(f"Directory del dataset non trovata: {data_root}")

if len(classes) == 0:
    raise ValueError(f"Nessuna classe trovata in {data_root}")

print(f'\nDirectory principale: {data_root}')
print(f'Numero di classi: {len(classes)}')
print(f'Classi: {classes}')


In [ ]:
# Conto quante immagini ci sono per ogni classe
class_counts = {}
for cls in classes:
    cls_path = os.path.join(data_root, cls)
    if os.path.exists(cls_path):
        # Cerco solo file immagine (png, jpg, jpeg)
        images = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        class_counts[cls] = len(images)

print('Numero di immagini per classe:')
for cls, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
    print(f'{cls}: {count} immagini')

print(f'\nTotale immagini: {sum(class_counts.values())}')


In [ ]:
# Visualizzo alcune immagini di esempio per vedere come sono fatte
def show_images_grid(data_root, classes, num_images=8):
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    axes = axes.flatten()

    for idx, cls in enumerate(classes[:num_images]):
        cls_path = os.path.join(data_root, cls)
        if os.path.exists(cls_path):
            images = [f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if images:
                # Prendo un'immagine a caso da questa classe
                img_path = os.path.join(cls_path, random.choice(images))
                img = Image.open(img_path)
                axes[idx].imshow(img)
                axes[idx].set_title(f'{cls}\n({class_counts[cls]} immagini)', fontsize=10)
                axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

show_images_grid(data_root, classes)


## 3. Data Augmentation

**Cosa facciamo in questa sezione:**
- Definizione delle trasformazioni per il training set (con augmentation) e validation/test set (senza augmentation)
- Visualizzazione degli effetti dell'augmentation su immagini di esempio

**Termini tecnici:**
- **Data Augmentation**: Tecnica per aumentare artificialmente la dimensione del dataset applicando trasformazioni casuali alle immagini
- **Transform Pipeline**: Sequenza di trasformazioni applicate alle immagini (resize, crop, flip, rotation, color jitter, normalization)
- **RandomResizedCrop**: Ritaglio casuale dell'immagine con ridimensionamento
- **ColorJitter**: Variazione casuale di luminosità, contrasto, saturazione e tonalità
- **Normalization**: Normalizzazione dei valori dei pixel usando media e deviazione standard di ImageNet


In [ ]:
# Definisco le trasformazioni per il training: uso data augmentation per avere più varietà
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Prima ridimensiono l'immagine
    transforms.RandomResizedCrop(224),  # Poi faccio un ritaglio casuale
    transforms.RandomHorizontalFlip(p=0.5),  # A volte capovolgo l'immagine orizzontalmente
    transforms.RandomRotation(15),  # Ruoto leggermente l'immagine
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Cambio un po' i colori
    transforms.ToTensor(),  # Converto in tensore PyTorch
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalizzo come ImageNet
])

# Per validation e test non uso augmentation, solo trasformazioni base
val_test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),  # Ritaglio centrato (non casuale)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
# Visualizzo come funziona l'augmentation: prendo un'immagine e la trasformo più volte
def show_augmentation_examples(data_root, classes, transform, num_examples=4):
    # Prendo una classe a caso
    cls = random.choice(classes)
    cls_path = os.path.join(data_root, cls)

    if os.path.exists(cls_path):
        images = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if images:
            img_path = os.path.join(cls_path, random.choice(images))
            original_img = Image.open(img_path)

            fig, axes = plt.subplots(1, num_examples + 1, figsize=(15, 3))

            # Mostro l'immagine originale
            axes[0].imshow(original_img)
            axes[0].set_title('Originale', fontsize=10)
            axes[0].axis('off')

            # Applico le trasformazioni più volte per vedere i risultati
            for i in range(1, num_examples + 1):
                transformed_img = transform(original_img)
                # De-normalizzo per poterla visualizzare (le immagini normalizzate non si vedono bene)
                img_np = transformed_img.permute(1, 2, 0).numpy()
                img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
                img_np = np.clip(img_np, 0, 1)
                axes[i].imshow(img_np)
                axes[i].set_title(f'Augmented {i}', fontsize=10)
                axes[i].axis('off')

            plt.suptitle(f'Esempi di augmentation - Classe: {cls}', fontsize=12)
            plt.tight_layout()
            plt.show()

show_augmentation_examples(data_root, classes, train_transform)


## 4. Dataset & DataLoader

**Cosa facciamo in questa sezione:**
- Creazione di una classe Dataset personalizzata che carica le immagini e applica le trasformazioni
- Split del dataset in train (70%), validation (15%) e test (15%) set
- Creazione dei DataLoader per il caricamento efficiente dei dati in batch durante il training

**Termini tecnici:**
- **Custom Dataset Class**: Classe che eredita da `torch.utils.data.Dataset` per gestire il caricamento personalizzato dei dati
- **Train/Validation/Test Split**: Divisione del dataset in tre subset per training, validazione e test finale
- **DataLoader**: Classe PyTorch che carica i dati in batch, con supporto per shuffling e multi-processing
- **Batch Size**: Numero di campioni processati insieme in un singolo forward pass
- **Shuffle**: Mescolamento casuale dei dati ad ogni epoca (solo per training set)


In [ ]:
# Creo una classe personalizzata per gestire il dataset
# Questa classe mi permette di caricare le immagini e applicare le trasformazioni
class FoodDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None):
        """
        Args:
            root_dir: cartella principale dove ci sono le classi
            classes: lista con i nomi delle classi
            transform: trasformazioni da applicare alle immagini
        """
        self.root_dir = root_dir
        self.classes = classes
        # Creo un dizionario per convertire nome classe -> numero
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        # E viceversa: numero -> nome classe
        self.idx_to_class = {idx: cls for idx, cls in enumerate(classes)}
        self.transform = transform

        # Raccolgo tutti i percorsi delle immagini e le loro etichette
        self.images = []
        self.labels = []

        for cls in classes:
            cls_path = os.path.join(root_dir, cls)
            if os.path.exists(cls_path):
                # Ordino le immagini per nome per garantire ordine consistente
                img_names = sorted([f for f in os.listdir(cls_path)
                                   if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                for img_name in img_names:
                    img_path = os.path.join(cls_path, img_name)
                    self.images.append(img_path)
                    self.labels.append(self.class_to_idx[cls])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]

        # Carico l'immagine
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f'Errore nel caricamento di {img_path}: {e}')
            # Se c'è un errore, ritorno un'immagine nera come fallback
            image = Image.new('RGB', (224, 224))

        # Applico le trasformazioni se ci sono
        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
# Creo il dataset base senza trasformazioni
full_dataset = FoodDataset(data_root, classes, transform=None)

# Verifico che il dataset non sia vuoto
dataset_size = len(full_dataset)
if dataset_size == 0:
    raise ValueError("Il dataset è vuoto! Verifica che ci siano immagini nelle cartelle delle classi.")

# Divido il dataset in train (70%), validation (15%) e test (15%)
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

# Verifico che le dimensioni siano valide
if train_size <= 0 or val_size <= 0 or test_size <= 0:
    raise ValueError(f"Dimensioni split non valide: Train={train_size}, Val={val_size}, Test={test_size}")

print(f"Totale immagini: {dataset_size}")
print(f"Split:\n  -  Train Set = {train_size} immagini\n  -  Validation Set = {val_size} immagini\n  -  Test Set = {test_size} immagini")

# Uso random_split per dividere il dataset in modo casuale
# Imposto un seed per riproducibilità
torch.manual_seed(42)
train_subset_base, val_subset_base, test_subset_base = random_split(full_dataset, [train_size, val_size, test_size])

# Salvo gli indici per poterli usare dopo con trasformazioni diverse
train_indices = list(train_subset_base.indices)
val_indices = list(val_subset_base.indices)
test_indices = list(test_subset_base.indices)

# Verifico che gli indici non siano vuoti
assert len(train_indices) > 0, "Train indices è vuoto!"
assert len(val_indices) > 0, "Val indices è vuoto!"
assert len(test_indices) > 0, "Test indices è vuoto!"

# Creo i dataset finali con le trasformazioni appropriate
# IMPORTANTE: L'ordine è garantito da sorted() nel FoodDataset
train_dataset = FoodDataset(data_root, classes, transform=train_transform)
val_dataset = FoodDataset(data_root, classes, transform=val_test_transform)
test_dataset = FoodDataset(data_root, classes, transform=val_test_transform)

# Verifico che i nuovi dataset abbiano la stessa lunghezza del dataset base
assert len(train_dataset) == dataset_size, f"Lunghezza train_dataset ({len(train_dataset)}) diversa da dataset_size ({dataset_size})"
assert len(val_dataset) == dataset_size, f"Lunghezza val_dataset ({len(val_dataset)}) diversa da dataset_size ({dataset_size})"
assert len(test_dataset) == dataset_size, f"Lunghezza test_dataset ({len(test_dataset)}) diversa da dataset_size ({dataset_size})"

# Verifico che gli indici siano validi (dentro il range)
max_index = dataset_size - 1
assert max(train_indices) <= max_index, f"Indice train fuori range: {max(train_indices)} > {max_index}"
assert max(val_indices) <= max_index, f"Indice val fuori range: {max(val_indices)} > {max_index}"
assert max(test_indices) <= max_index, f"Indice test fuori range: {max(test_indices)} > {max_index}"

# Creo i subset usando gli indici salvati
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)
test_subset = Subset(test_dataset, test_indices)

# Verifico che i subset non siano vuoti
assert len(train_subset) > 0, f"Train subset è vuoto! Lunghezza: {len(train_subset)}"
assert len(val_subset) > 0, f"Val subset è vuoto! Lunghezza: {len(val_subset)}"
assert len(test_subset) > 0, f"Test subset è vuoto! Lunghezza: {len(test_subset)}"

print(f"Dimensioni subset: Train={len(train_subset)}, Val={len(val_subset)}, Test={len(test_subset)}")
print("Dataset con trasformazioni appropriate creati con successo!")

In [ ]:
# Creo i DataLoader che mi permettono di caricare i dati a batch
batch_size = 32
# num_workers=0 su Windows/Mac per evitare problemi, 2 su Linux
num_workers = 0 if platform.system() in ['Windows', 'Darwin'] else 2

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f'Batch size: {batch_size}')
print(f'Numero di batch nel train loader: {len(train_loader)}')
print(f'Numero di batch nel validation loader: {len(val_loader)}')
print(f'Numero di batch nel test loader: {len(test_loader)}')

## 5. Transfer Learning

**Cosa facciamo in questa sezione:**
- Caricamento di un modello pre-addestrato (ResNet18) su ImageNet
- Sostituzione del classificatore finale per adattarlo al nostro problema (14 classi invece di 1000)
- Congelamento del backbone (feature extractor) mantenendo allenabile solo il classificatore finale
- Analisi del numero di parametri allenabili vs congelati

**Termini tecnici:**
- **Transfer Learning**: Tecnica che riutilizza un modello pre-addestrato su un dataset grande (ImageNet) per un nuovo task
- **Pre-trained Model**: Modello già addestrato su un dataset grande, le cui feature possono essere riutilizzate
- **Backbone/Feature Extractor**: Parte convolutiva del modello che estrae feature dalle immagini
- **Classifier Head**: Ultimo layer fully-connected che mappa le feature alle classi
- **Freezing**: Congelamento dei parametri (requires_grad=False) per evitare l'aggiornamento durante il training
- **Fine-tuning**: Processo di adattamento di un modello pre-addestrato al nuovo task


In [ ]:
# Creo una funzione per creare il modello ResNet18 con transfer learning
def create_model(num_classes, freeze_backbone=True):
    """
    Creo un modello ResNet18 già addestrato su ImageNet e lo adatto al mio problema

    Args:
        num_classes: quante classi devo classificare (nel mio caso 14)
        freeze_backbone: se True, non alleno i layer convolutivi (solo l'ultimo layer)
    """
    # Carico ResNet18 già addestrato su ImageNet
    model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

    # Se voglio congelare il backbone, dico a PyTorch di non calcolare i gradienti
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Sostituisco l'ultimo layer (che era per 1000 classi ImageNet) con uno per le mie classi
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)

    # L'ultimo layer deve essere allenabile anche se il resto è congelato
    if freeze_backbone:
        for param in model.fc.parameters():
            param.requires_grad = True

    return model

# Creo il modello per il baseline (con backbone congelato)
num_classes = len(classes)
model = create_model(num_classes, freeze_backbone=True)
model = model.to(device)

# Conto quanti parametri ho in totale e quanti posso allenare
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Numero di classi: {num_classes}')
print(f'Parametri totali: {total_params:,}')
print(f'Parametri allenabili: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)')
print(f'Parametri congelati: {total_params - trainable_params:,} ({(total_params - trainable_params)/total_params*100:.2f}%)')


## 6. Baseline Training

**Cosa facciamo in questa sezione:**
- Definizione delle funzioni di training e validation
- Setup dell'ottimizzatore (Adam), loss function (CrossEntropyLoss) e early stopping
- Training del modello con backbone congelato (solo il classificatore viene allenato)
- Monitoraggio di loss e accuracy su train e validation set
- Salvataggio del miglior modello basato sulla validation accuracy

**Termini tecnici:**
- **Baseline Model**: Modello iniziale di riferimento, tipicamente con backbone congelato
- **Training Loop**: Ciclo che itera sulle epoche, eseguendo forward pass, backward pass e aggiornamento dei pesi
- **Forward Pass**: Calcolo delle predizioni del modello dato un input
- **Backward Pass**: Calcolo dei gradienti tramite backpropagation
- **Optimizer**: Algoritmo che aggiorna i pesi del modello (es. Adam, SGD)
- **Learning Rate**: Iperparametro che controlla la dimensione degli step di aggiornamento
- **Loss Function**: Funzione che misura l'errore tra predizioni e ground truth (CrossEntropyLoss per classificazione)
- **Early Stopping**: Tecnica per fermare il training quando la validation accuracy non migliora più
- **Validation Accuracy**: Accuratezza del modello sul validation set, usata per monitorare overfitting

In [ ]:
# Definisco le funzioni per il training e la validazione
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Alleno il modello per un'epoca completa"""
    model.train()  # Metto il modello in modalità training
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        # Sposto immagini e etichette sul dispositivo (CPU o GPU)
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass: calcolo le predizioni e la loss
        optimizer.zero_grad()  # Azzero i gradienti del passo precedente
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass: calcolo i gradienti e aggiorno i pesi
        loss.backward()
        optimizer.step()

        # Tengo traccia delle statistiche
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)  # Prendo la classe predetta
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calcolo la media della loss e l'accuracy per questa epoca
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate(model, val_loader, criterion, device):
    """Valido il modello senza aggiornare i pesi"""
    model.eval()  # Metto il modello in modalità evaluation
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # Non calcolo i gradienti (risparmio memoria)
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

# Preparo tutto per il training baseline
criterion = nn.CrossEntropyLoss()  # Uso questa loss per la classificazione
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Uso Adam come ottimizzatore

# Imposto l'early stopping: se non miglioro per 5 epoche, smetto
best_val_acc = 0.0
patience = 5
patience_counter = 0
best_model_state = None

# Tengo traccia della storia del training per vedere come va
train_losses = []
train_accs = []
val_losses = []
val_accs = []

num_epochs = 20

print('Inizio training baseline...')
print(f'Numero di epoche: {num_epochs}')
print(f'Learning rate: {optimizer.param_groups[0]["lr"]}')
print(f'Early stopping patience: {patience}')
print('-' * 50)


In [ ]:
# Ciclo di training: alleno per diverse epoche
for epoch in range(num_epochs):
    # Alleno il modello su tutto il training set
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # Valido il modello sul validation set
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Salvo le statistiche per poi farci i grafici
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Stampo i risultati di questa epoca
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

    # Controllo se ho migliorato e applico early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        # Salvo il miglior modello finora
        best_model_state = model.state_dict().copy()
        print(f'✓ Nuovo miglior modello! Val Acc: {best_val_acc:.2f}%')
    else:
        patience_counter += 1
        print(f'  Nessun miglioramento ({patience_counter}/{patience})')

    # Se non miglioro da troppo tempo, smetto
    if patience_counter >= patience:
        print(f'\nEarly stopping attivato dopo {epoch+1} epoche')
        break

    print('-' * 50)

# Carico il miglior modello che ho salvato
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f'\nModello baseline ottimale caricato. Best Val Acc: {best_val_acc:.2f}%')


In [ ]:
# Visualizzo i grafici per vedere come è andato il training
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Grafico della loss (dovrebbe diminuire)
ax1.plot(train_losses, label='Train Loss', marker='o')
ax1.plot(val_losses, label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training e Validation Loss - Baseline')
ax1.legend()
ax1.grid(True)

# Grafico dell'accuracy (dovrebbe aumentare)
ax2.plot(train_accs, label='Train Acc', marker='o')
ax2.plot(val_accs, label='Val Acc', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training e Validation Accuracy - Baseline')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f'\nRisultati Baseline:')
print(f'Best Validation Accuracy: {best_val_acc:.2f}%')
if len(train_accs) > 0:
    print(f'Final Train Accuracy: {train_accs[-1]:.2f}%')
    print(f'Final Validation Accuracy: {val_accs[-1]:.2f}%')
else:
    print('Training non completato - nessuna epoca eseguita')


## 7. Fine-tuning & Regularization

**Cosa facciamo in questa sezione:**
- Scongelamento degli ultimi layer convolutivi del backbone per permettere l'adattamento alle nostre immagini
- Aggiunta di tecniche di regularizzazione (Dropout e Weight Decay) per prevenire overfitting
- Training del modello con learning rate più basso (i layer sono già quasi ottimali)
- Confronto delle performance con il baseline per valutare il miglioramento

**Termini tecnici:**
- **Fine-tuning**: Processo di allenamento di layer aggiuntivi del modello pre-addestrato
- **Unfreezing**: Abilitazione del training di layer precedentemente congelati
- **Regularization**: Tecniche per prevenire overfitting e migliorare la generalizzazione
- **Dropout**: Tecnica di regularizzazione che disattiva casualmente neuroni durante il training
- **Weight Decay (L2 Regularization)**: Penalizzazione dei pesi grandi per evitare overfitting
- **Learning Rate Scheduling**: Riduzione del learning rate durante il training (qui usiamo un LR più basso per fine-tuning)


In [ ]:
# Per il fine tuning, scongelo alcuni layer del backbone così posso allenarli
def unfreeze_layers(model, num_layers_to_unfreeze=2):
    """
    Scongelo gli ultimi layer del ResNet così posso allenarli
    """
    # ResNet18 ha 4 layer (layer1, layer2, layer3, layer4)
    # Scongelo gli ultimi 2 (layer4 e layer3)
    layers_to_unfreeze = ['layer4', 'layer3']

    for name, param in model.named_parameters():
        # Scongelo gli ultimi layer convolutivi
        if any(layer in name for layer in layers_to_unfreeze[:num_layers_to_unfreeze]):
            param.requires_grad = True
        # Il layer fc è già allenabile
        elif 'fc' in name:
            param.requires_grad = True

    return model

# Creo un nuovo modello per il fine tuning
model_ft = create_model(num_classes, freeze_backbone=True)
model_ft = unfreeze_layers(model_ft, num_layers_to_unfreeze=2)
model_ft = model_ft.to(device)

# Conto quanti parametri posso allenare ora
total_params_ft = sum(p.numel() for p in model_ft.parameters())
trainable_params_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)

print('Modello per Fine Tuning:')
print(f'Parametri totali: {total_params_ft:,}')
print(f'Parametri allenabili: {trainable_params_ft:,} ({trainable_params_ft/total_params_ft*100:.2f}%)')
print(f'Parametri congelati: {total_params_ft - trainable_params_ft:,} ({(total_params_ft - trainable_params_ft)/total_params_ft*100:.2f}%)')


In [ ]:
# Aggiungo dropout al classifier per evitare overfitting (regularizzazione)
# Modifico l'ultimo layer per includere dropout prima del layer lineare
num_features = model_ft.fc.in_features
model_ft.fc = nn.Sequential(
    nn.Dropout(0.5),  # Dropout: durante il training, metto a zero il 50% dei neuroni casualmente
    nn.Linear(num_features, num_classes)
)
model_ft = model_ft.to(device)

print('Modello con dropout aggiunto al classifier')
print(model_ft.fc)


In [ ]:
# Preparo tutto per il fine tuning con regularizzazione
criterion_ft = nn.CrossEntropyLoss()
# Uso un learning rate più basso per il fine tuning (i layer sono già quasi buoni)
optimizer_ft = optim.Adam(model_ft.parameters(), lr=0.0001, weight_decay=1e-4)  # Weight decay aiuta contro overfitting

# Reset delle variabili per early stopping
best_val_acc_ft = 0.0
patience = 5  # Definisco patience anche per il fine-tuning
patience_counter_ft = 0
best_model_state_ft = None

# Tengo traccia della storia del training
train_losses_ft = []
train_accs_ft = []
val_losses_ft = []
val_accs_ft = []

num_epochs_ft = 20

print('Inizio Fine Tuning con Regularizzazione...')
print(f'Numero di epoche: {num_epochs_ft}')
print(f'Learning rate: {optimizer_ft.param_groups[0]["lr"]}')
print(f'Weight decay: {optimizer_ft.param_groups[0]["weight_decay"]}')
print(f'Dropout: 0.5')
print('-' * 50)


In [ ]:
# Ciclo di training per il fine tuning
for epoch in range(num_epochs_ft):
    # Alleno il modello
    train_loss, train_acc = train_epoch(model_ft, train_loader, criterion_ft, optimizer_ft, device)

    # Valido il modello
    val_loss, val_acc = validate(model_ft, val_loader, criterion_ft, device)

    # Salvo le statistiche
    train_losses_ft.append(train_loss)
    train_accs_ft.append(train_acc)
    val_losses_ft.append(val_loss)
    val_accs_ft.append(val_acc)

    # Stampo i risultati
    print(f'Epoch [{epoch+1}/{num_epochs_ft}]')
    print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

    # Controllo early stopping
    if val_acc > best_val_acc_ft:
        best_val_acc_ft = val_acc
        patience_counter_ft = 0
        best_model_state_ft = model_ft.state_dict().copy()
        print(f'✓ Nuovo miglior modello! Val Acc: {best_val_acc_ft:.2f}%')
    else:
        patience_counter_ft += 1
        print(f'  Nessun miglioramento ({patience_counter_ft}/{patience})')

    if patience_counter_ft >= patience:
        print(f'\nEarly stopping attivato dopo {epoch+1} epoche')
        break

    print('-' * 50)

# Carico il miglior modello
if best_model_state_ft is not None:
    model_ft.load_state_dict(best_model_state_ft)
    print(f'\nModello fine-tuned ottimale caricato. Best Val Acc: {best_val_acc_ft:.2f}%')


In [ ]:
# Visualizzo i grafici del fine tuning
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Grafico della loss
ax1.plot(train_losses_ft, label='Train Loss', marker='o')
ax1.plot(val_losses_ft, label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training e Validation Loss - Fine Tuning')
ax1.legend()
ax1.grid(True)

# Grafico dell'accuracy
ax2.plot(train_accs_ft, label='Train Acc', marker='o')
ax2.plot(val_accs_ft, label='Val Acc', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training e Validation Accuracy - Fine Tuning')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f'\nRisultati Fine Tuning:')
print(f'Best Validation Accuracy: {best_val_acc_ft:.2f}%')
if len(train_accs_ft) > 0:
    print(f'Final Train Accuracy: {train_accs_ft[-1]:.2f}%')
    print(f'Final Validation Accuracy: {val_accs_ft[-1]:.2f}%')
else:
    print('Training non completato - nessuna epoca eseguita')

# Confronto con il baseline per vedere se ho migliorato
if 'best_val_acc' in globals() and len(train_accs_ft) > 0:
    print(f'\nConfronto:')
    print(f'Baseline - Best Val Acc: {best_val_acc:.2f}%')
    print(f'Fine Tuning - Best Val Acc: {best_val_acc_ft:.2f}%')
    print(f'Miglioramento: {best_val_acc_ft - best_val_acc:.2f}%')


## 8. Model Evaluation

**Cosa facciamo in questa sezione:**
- Valutazione finale del modello sul test set (dati mai visti durante il training)
- Calcolo della confusion matrix per analizzare gli errori per classe
- Visualizzazione di predizioni campione con confidence scores
- Calcolo dell'accuracy per ogni singola classe

**Termini tecnici:**
- **Model Evaluation**: Valutazione delle performance del modello su dati di test
- **Test Set**: Dataset separato usato solo per la valutazione finale, mai visto durante training/validation
- **Confusion Matrix**: Matrice che mostra le predizioni corrette e sbagliate per ogni classe
- **Accuracy**: Percentuale di predizioni corrette sul totale
- **Per-class Accuracy**: Accuratezza calcolata separatamente per ogni classe
- **Confidence Score**: Probabilità associata alla predizione del modello
- **Ground Truth**: Etichette vere/corrette dei dati


In [ ]:
# Definisco una funzione per testare il modello sul test set
def test_model(model, test_loader, device, classes):
    """Valuto il modello sul test set e salvo tutte le predizioni"""
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Salvo tutte le predizioni e le etichette vere per la matrice di confusione
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * correct / total
    return accuracy, all_preds, all_labels

# Testo il modello fine-tuned sul test set
test_acc, test_preds, test_labels = test_model(model_ft, test_loader, device, classes)

print(f'\nTest Finale - Accuracy: {test_acc:.2f}%')
print(f'Numero di immagini testate: {len(test_preds)}')


In [ ]:
# Creo la matrice di confusione per vedere dove il modello sbaglia
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Matrice di Confusione - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Calcolo l'accuracy per ogni classe singolarmente
print('\nAccuracy per classe:')
class_correct = cm.diagonal()  # Predizioni corrette per ogni classe
class_total = cm.sum(axis=1)  # Totale immagini per ogni classe
for i, cls in enumerate(classes):
    if class_total[i] > 0:
        cls_acc = 100 * class_correct[i] / class_total[i]
        print(f'{cls}: {cls_acc:.2f}% ({class_correct[i]}/{class_total[i]})')


In [ ]:
# Visualizzo alcune predizioni per vedere come si comporta il modello
def show_predictions(model, test_loader, classes, device, num_samples=8):
    model.eval()

    # Prendo un batch dal test loader
    images, labels = next(iter(test_loader))
    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)  # Calcolo le probabilità

    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    axes = axes.flatten()

    for idx in range(min(num_samples, len(images))):
        img = images[idx].cpu()
        # De-normalizzo per visualizzare l'immagine
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)

        true_label = classes[labels[idx].item()]
        pred_label = classes[predicted[idx].item()]
        confidence = probabilities[idx][predicted[idx]].item() * 100  # Quanto è sicuro il modello

        axes[idx].imshow(img_np)
        # Verde se ha indovinato, rosso se ha sbagliato
        color = 'green' if labels[idx] == predicted[idx] else 'red'
        axes[idx].set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)',
                           color=color, fontsize=10)
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

show_predictions(model_ft, test_loader, classes, device)


## 9. Conclusions & Results Summary

**Cosa facciamo in questa sezione:**
- Analisi e discussione dei risultati ottenuti
- Confronto tra baseline e fine-tuned model
- Riflessioni sulle tecniche utilizzate e sul loro impatto

**Termini tecnici:**
- **Baseline vs Fine-tuned Comparison**: Confronto tra modello iniziale e modello ottimizzato
- **Performance Metrics**: Metriche usate per valutare il modello (accuracy, loss)
- **Generalization**: Capacità del modello di performare bene su dati nuovi e non visti

### Analisi dei Risultati

Il progetto ha implementato con successo un sistema di classificazione di immagini nel settore food utilizzando tecniche di deep learning con PyTorch.

**Data Augmentation:** L'implementazione di tecniche di augmentation (random crop, flip, rotation, color jitter) ha contribuito ad aumentare la variabilità del dataset di training, migliorando la capacità di generalizzazione del modello.

**Transfer Learning:** L'utilizzo di ResNet18 pre-addestrato su ImageNet ha permesso di sfruttare feature già apprese su un dataset molto grande, riducendo significativamente il tempo di training e migliorando le performance iniziali.

**Fine Tuning:** Lo scongelamento degli ultimi layer convolutivi ha permesso al modello di adattarsi meglio alle caratteristiche specifiche del dataset food, migliorando ulteriormente le performance rispetto al baseline.

**Regularizzazione:** L'aggiunta di dropout e weight decay ha aiutato a prevenire l'overfitting, garantendo una migliore generalizzazione sul test set.

**Risultati:** Il modello finale ha raggiunto un'accuracy soddisfacente sul test set, dimostrando l'efficacia dell'approccio utilizzato. Il confronto tra baseline e fine-tuned model mostra un miglioramento significativo, confermando l'importanza delle tecniche di ottimizzazione implementate.
